In [ ]:
from pathlib import Path
import sys
import pickle
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data.transaction_utils import build_transactions

win_id = 1435

binary_path = ROOT / "data/stream/swat/window_binary_data" / f"window_{win_id}_binary.csv"
save_path = ROOT / "data/stream/swat/window_transactions" / f"window_{win_id}_transactions.pkl"

df_binary = pd.read_csv(binary_path)
transactions = build_transactions(df_binary, lag=2)

with open(save_path, "wb") as f:
    pickle.dump(transactions, f)

print("saved:", save_path)
print("num_transactions:", len(transactions))
print("first_transaction:", transactions[0][:10] if len(transactions) > 0 else [])

In [ ]:
from pathlib import Path
import pickle
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

# 1) 读取窗口索引信息
df_win = pd.read_csv(ROOT / "data/stream/swat/windows_mixed.csv")
row = df_win[df_win["window_id"] == win_id].iloc[0]
start_idx = int(row["start_idx"])
end_idx = int(row["end_idx"])

# 2) 读取全局标签
y_df = pd.read_csv(ROOT / "data/processed/swat/y_filled.csv")
if y_df.shape[1] == 1:
    y_all = y_df.iloc[:, 0].astype(int).reset_index(drop=True)
else:
    y_all = y_df["label"].astype(int).reset_index(drop=True)

window_labels = y_all.iloc[start_idx:end_idx].reset_index(drop=True)

# 3) 读取刚保存的 transactions
tx_path = ROOT / "data/stream/swat/window_transactions" / f"window_{win_id}_transactions.pkl"
with open(tx_path, "rb") as f:
    transactions = pickle.load(f)

assert len(window_labels) == len(transactions), (len(window_labels), len(transactions))

# 4) 生成 labeled transactions
labeled_transactions = []
for tx, y in zip(transactions, window_labels):
    tx2 = list(tx) + (["LABEL_ATTACK"] if int(y) == 1 else ["LABEL_NORMAL"])
    labeled_transactions.append(tx2)

# 5) 保存
save_path = ROOT / "data/stream/swat/window_labeled_transactions" / f"window_{win_id}_labeled_transactions.pkl"
with open(save_path, "wb") as f:
    pickle.dump(labeled_transactions, f)

print("start_idx:", start_idx, "end_idx:", end_idx)
print("num_transactions:", len(transactions))
print("label_counts:\n", window_labels.value_counts().sort_index())
print("saved:", save_path)
print("last_token_of_first_tx:", labeled_transactions[0][-1])

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

in_path = ROOT / "data/stream/swat/window_labeled_transactions" / f"window_{win_id}_labeled_transactions.pkl"
out_path = ROOT / "data/stream/swat/window_labeled_onehot" / f"window_{win_id}_labeled_onehot.csv"

with open(in_path, "rb") as f:
    labeled_transactions = pickle.load(f)

te = TransactionEncoder()
onehot = te.fit(labeled_transactions).transform(labeled_transactions)

df_onehot = pd.DataFrame(onehot, columns=te.columns_)
df_onehot.to_csv(out_path, index=False)

print("saved:", out_path)
print("shape:", df_onehot.shape)
print("label_cols:", [c for c in df_onehot.columns if "LABEL_" in c])
print(df_onehot[[c for c in df_onehot.columns if "LABEL_" in c]].sum())

In [ ]:
from pathlib import Path
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

onehot_path = ROOT / "data/stream/swat/window_labeled_onehot" / f"window_{win_id}_labeled_onehot.csv"
save_path = ROOT / "data/stream/swat/window_freq_items" / f"window_{win_id}_freq_items.csv"

df_onehot = pd.read_csv(onehot_path)

freq_items = fpgrowth(
    df_onehot,
    min_support=0.1,
    use_colnames=True,
    max_len=2
).sort_values(["support"], ascending=False).reset_index(drop=True)

freq_items["itemset_len"] = freq_items["itemsets"].apply(len)
freq_items.to_csv(save_path, index=False)

print("saved:", save_path)
print("num_freq_items:", len(freq_items))
print(freq_items.head(10))

In [ ]:
from pathlib import Path
import pandas as pd
from mlxtend.frequent_patterns import association_rules

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

save_path = ROOT / "data/stream/swat/window_label_rules" / f"window_{win_id}_label_rules.csv"

rules = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.6
).copy()

rules = rules[
    (rules["consequents"].apply(lambda x: len(x) == 1)) &
    (rules["consequents"].apply(lambda x: list(x)[0] in ["LABEL_ATTACK", "LABEL_NORMAL"]))
].copy()

rules["antecedent_str"] = rules["antecedents"].apply(lambda x: " & ".join(sorted(list(x))))
rules["consequent_str"] = rules["consequents"].apply(lambda x: list(x)[0])

keep_cols = [
    "antecedent_str", "consequent_str",
    "support", "confidence", "lift"
]
rules2 = rules[keep_cols].sort_values(
    ["consequent_str", "confidence", "lift", "support"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

rules2.to_csv(save_path, index=False)

attack_rules = rules2[rules2["consequent_str"] == "LABEL_ATTACK"].reset_index(drop=True)
normal_rules = rules2[rules2["consequent_str"] == "LABEL_NORMAL"].reset_index(drop=True)

print("saved:", save_path)
print("num_attack_rules:", len(attack_rules))
print("num_normal_rules:", len(normal_rules))
print("\n[attack top 10]")
print(attack_rules.head(10))
print("\n[normal top 10]")
print(normal_rules.head(10))

In [ ]:
from pathlib import Path
import pandas as pd
from mlxtend.frequent_patterns import association_rules

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

rules_all = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.0
).copy()

rules_all = rules_all[
    (rules_all["consequents"].apply(lambda x: len(x) == 1)) &
    (rules_all["consequents"].apply(lambda x: list(x)[0] in ["LABEL_ATTACK", "LABEL_NORMAL"]))
].copy()

rules_all["antecedent_str"] = rules_all["antecedents"].apply(lambda x: " & ".join(sorted(list(x))))
rules_all["consequent_str"] = rules_all["consequents"].apply(lambda x: list(x)[0])

normal_all = rules_all[rules_all["consequent_str"] == "LABEL_NORMAL"].copy()
normal_all = normal_all.sort_values(
    ["confidence", "lift", "support"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("num_normal_candidates:", len(normal_all))
if len(normal_all) > 0:
    print("max_normal_confidence:", normal_all["confidence"].max())
    print(normal_all[["antecedent_str", "support", "confidence", "lift"]].head(20))
else:
    print("no normal candidates at all")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

p_normal = 310 / 2000  # 当前窗口正常比例
save_path = ROOT / "data/stream/swat/window_label_rules" / f"window_{win_id}_normal_candidates_relaxed.csv"

normal_ranked = normal_all.copy()
normal_ranked["prior_normal"] = p_normal
normal_ranked["conf_gain"] = normal_ranked["confidence"] - p_normal
normal_ranked["score_relaxed"] = normal_ranked["conf_gain"] + 0.1 * (normal_ranked["lift"] - 1.0)

keep_cols = ["antecedent_str", "support", "confidence", "lift", "prior_normal", "conf_gain", "score_relaxed"]
normal_ranked = normal_ranked[keep_cols].sort_values(
    ["score_relaxed", "confidence", "lift", "support"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

normal_ranked.to_csv(save_path, index=False)

print("saved:", save_path)
print("p_normal:", p_normal)
print(normal_ranked.head(10))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

pool_dir = ROOT / "data/stream/swat/window_rule_pools"
pool_dir.mkdir(parents=True, exist_ok=True)

# 1) 取 top attack / top relaxed normal
attack_top = attack_rules.head(10).copy()
normal_top = normal_ranked.head(3).copy()

def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    w = confidence_to_weight(conf)
    return float(np.clip(w, 0.0, wmax))

# 2) 补齐字段
attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
attack_top["target_label"] = 1

normal_top["consequent_str"] = "LABEL_NORMAL"
normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
normal_top["weight"] = normal_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
normal_top["target_label"] = 0

mixed_rule_pool = pd.concat([
    attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
], axis=0, ignore_index=True)

save_path = pool_dir / f"window_{win_id}_mixed_rule_pool.csv"
mixed_rule_pool.to_csv(save_path, index=False)

print("saved:", save_path)
print("num_rules:", len(mixed_rule_pool))
print("label_counts:")
print(mixed_rule_pool["target_label"].value_counts())
print(mixed_rule_pool[["formula", "confidence", "weight", "target_label"]])

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

pool_dir = ROOT / "data/stream/swat/window_rule_pools"
pool_dir.mkdir(parents=True, exist_ok=True)

attack_top = attack_rules.head(10).copy()
normal_top = normal_ranked.head(3).copy()

def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    w = confidence_to_weight(conf)
    return float(np.clip(w, 0.0, wmax))

def relaxed_normal_weight(score_relaxed, wmin=0.3, scale=30.0, wmax=3.0):
    w = scale * float(score_relaxed)
    return float(np.clip(w, wmin, wmax))

attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
attack_top["target_label"] = 1

normal_top["consequent_str"] = "LABEL_NORMAL"
normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
normal_top["weight"] = normal_top["score_relaxed"].apply(lambda x: relaxed_normal_weight(x, wmin=0.3, scale=30.0, wmax=3.0))
normal_top["target_label"] = 0

mixed_rule_pool = pd.concat([
    attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
], axis=0, ignore_index=True)

save_path = pool_dir / f"window_{win_id}_mixed_rule_pool.csv"
mixed_rule_pool.to_csv(save_path, index=False)

print("saved:", save_path)
print(mixed_rule_pool[["formula", "confidence", "weight", "target_label"]])

In [ ]:
from pathlib import Path
import sys
import pickle
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.rl.state_utils import build_initial_rule_state

win_id = 1435

pool_path = ROOT / "data/stream/swat/window_rule_pools" / f"window_{win_id}_mixed_rule_pool.csv"
save_path = ROOT / "data/stream/swat/window_rule_states" / f"window_{win_id}_rule_state.pkl"

mixed_rule_pool = pd.read_csv(pool_path)
state_dict = build_initial_rule_state(mixed_rule_pool)

with open(save_path, "wb") as f:
    pickle.dump(state_dict, f)

print("saved:", save_path)
print("keys:", list(state_dict.keys()))
print("num_rules:", state_dict["num_rules"])
print("active_sum:", state_dict["active_mask"].sum())
print("weights:", state_dict["weights"])
print("target_labels:", state_dict["target_labels"])

In [ ]:
from pathlib import Path
import sys
import pickle
import numpy as np

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.rl.simple_env import SimpleRuleEnv
from src.rl.reward_utils import compute_reward

win_id = 1435
state_path = ROOT / "data/stream/swat/window_rule_states" / f"window_{win_id}_rule_state.pkl"

with open(state_path, "rb") as f:
    state_dict = pickle.load(f)

# 1) 初始环境
env = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=20)
state0 = env.reset()

init_reward = compute_reward(
    env.active_mask,
    env.weights,
    env.rule_scores,
    env.initial_weights,
    env.target_labels
)

print("state_dim:", state0.shape)
print("state0:", state0)
print("init_reward:", init_reward)

# 2) 找第一条 normal 规则，测试 disable
normal_idx = int(np.where(env.target_labels == 0)[0][0])

env_n = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=20)
env_n.reset()
next_state_n, reward_n, done_n, info_n = env_n.step(normal_idx, 1)   # 1 = disable

print("\n[disable normal once]")
print("normal_idx:", normal_idx)
print("delta_reward:", reward_n)
print("info:", info_n)

# 3) 找第一条 attack 规则，测试 disable
attack_idx = int(np.where(env.target_labels == 1)[0][0])

env_a = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=20)
env_a.reset()
next_state_a, reward_a, done_a, info_a = env_a.step(attack_idx, 1)   # 1 = disable

print("\n[disable attack once]")
print("attack_idx:", attack_idx)
print("delta_reward:", reward_a)
print("info:", info_a)

In [ ]:
import numpy as np
import pickle
from pathlib import Path

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

state_path = ROOT / "data/stream/swat/window_rule_states" / f"window_{win_id}_rule_state.pkl"

with open(state_path, "rb") as f:
    state_dict = pickle.load(f)

active_mask = np.array(state_dict["active_mask"], dtype=np.float32)
weights = np.array(state_dict["weights"], dtype=np.float32)
rule_scores = np.array(state_dict["rule_scores"], dtype=np.float32)
target_labels = np.array(state_dict["target_labels"], dtype=np.int64)
num_rules = int(state_dict["num_rules"])

def build_ws_state(active_mask, weights, rule_scores, target_labels, rule_idx):
    active_weights = weights[active_mask > 0.5]
    if len(active_weights) == 0:
        mean_w = std_w = min_w = max_w = 0.0
    else:
        mean_w = float(active_weights.mean())
        std_w = float(active_weights.std())
        min_w = float(active_weights.min())
        max_w = float(active_weights.max())

    x = np.array([
        float(active_mask.mean()),              # 全局激活比例
        mean_w,                                 # 全局平均权重
        std_w,                                  # 权重标准差
        min_w,                                  # 最小权重
        max_w,                                  # 最大权重
        float(active_mask.sum()),               # 当前激活规则数
        float(num_rules),                       # 总规则数
        float(rule_idx / max(1, num_rules-1)),  # rule_idx归一化位置
        float(active_mask[rule_idx]),           # 当前规则是否激活
        float(weights[rule_idx]),               # 当前规则权重
        float(rule_scores[rule_idx]),           # 当前规则分数
        float(target_labels[rule_idx]),         # 当前规则target_label
    ], dtype=np.float32)
    return x

X_ws = np.stack([
    build_ws_state(active_mask, weights, rule_scores, target_labels, i)
    for i in range(num_rules)
], axis=0)

# attack=1 -> keep(0), normal=0 -> disable(1)
y_ws = np.array([0 if t == 1 else 1 for t in target_labels], dtype=np.int64)

num_keep = int((y_ws == 0).sum())
num_disable = int((y_ws == 1).sum())

print("X_ws.shape:", X_ws.shape)
print("y_ws:", y_ws)
print("num_keep:", num_keep)
print("num_disable:", num_disable)
print("first_state:", X_ws[0])
print("last_state:", X_ws[-1])

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.rl.ac_model import ActorCriticNet

win_id = 1435
save_path = ROOT / "outputs/models" / f"window_{win_id}_actor_warmstart.pth"
save_path.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X_ws, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y_ws, dtype=torch.long).to(device)

num_keep = int((y_ws == 0).sum())
num_disable = int((y_ws == 1).sum())

# keep类少权重低，disable类少权重高
class_weights = torch.tensor(
    [1.0 / max(num_keep, 1), 1.0 / max(num_disable, 1)],
    dtype=torch.float32
).to(device)

model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights)

def get_logits(output):
    if isinstance(output, tuple) or isinstance(output, list):
        return output[0]
    if isinstance(output, dict):
        for k in ["policy_logits", "logits", "actor_logits"]:
            if k in output:
                return output[k]
    return output

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    out = model(X_tensor)
    logits = get_logits(out)
    loss = criterion(logits, y_tensor)

    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    logits = get_logits(model(X_tensor))
    pred = logits.argmax(dim=1)
    acc = (pred == y_tensor).float().mean().item()

torch.save(model.state_dict(), save_path)

print("saved:", save_path)
print("class_weights:", class_weights.detach().cpu().numpy())
print("train_pred:", pred.detach().cpu().numpy())
print("train_acc:", acc)

In [ ]:
from pathlib import Path
import sys
import pickle
import numpy as np
import pandas as pd
import torch

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.rl.ac_model import ActorCriticNet
from src.rl.simple_env import SimpleRuleEnv

win_id = 1435

state_path = ROOT / "data/stream/swat/window_rule_states" / f"window_{win_id}_rule_state.pkl"
model_path = ROOT / "outputs/models" / f"window_{win_id}_actor_warmstart.pth"
metrics_path = ROOT / "outputs/logs" / f"window_{win_id}_metrics.csv"
trace_path = ROOT / "outputs/logs" / f"window_{win_id}_greedy_trace.csv"

with open(state_path, "rb") as f:
    state_dict = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

def get_logits(output):
    if isinstance(output, tuple) or isinstance(output, list):
        return output[0]
    if isinstance(output, dict):
        for k in ["policy_logits", "logits", "actor_logits"]:
            if k in output:
                return output[k]
    return output

env = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=100)
state = env.reset()

records = []
total_reward = 0.0
num_rules = int(state_dict["num_rules"])
target_labels = np.array(state_dict["target_labels"])

for rule_idx in range(num_rules):
    x = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = get_logits(model(x))
        action = int(torch.argmax(logits, dim=1).item())   # 0=keep, 1=disable

    next_state, reward, done, info = env.step(rule_idx, action)
    total_reward += float(reward)

    target_label = int(target_labels[rule_idx])
    is_correct = int((target_label == 1 and action == 0) or (target_label == 0 and action == 1))

    records.append({
        "rule_idx": rule_idx,
        "action": action,
        "reward": float(reward),
        "target_label": target_label,
        "is_correct": is_correct,
        "weight": float(info["weight"]),
        "rule_score": float(info["rule_score"]),
        "active_after": int(info["active"]),
    })

    state = next_state

trace_df = pd.DataFrame(records)
trace_df.to_csv(trace_path, index=False)

attack_mask = trace_df["target_label"] == 1
normal_mask = trace_df["target_label"] == 0

attack_keep_rate = float((trace_df.loc[attack_mask, "action"] == 0).mean()) if attack_mask.sum() > 0 else np.nan
normal_disable_rate = float((trace_df.loc[normal_mask, "action"] == 1).mean()) if normal_mask.sum() > 0 else np.nan
selection_accuracy = float(trace_df["is_correct"].mean())

metrics_df = pd.DataFrame([{
    "window_id": win_id,
    "num_rules": num_rules,
    "num_attack_rules": int(attack_mask.sum()),
    "num_normal_rules": int(normal_mask.sum()),
    "attack_keep_rate": attack_keep_rate,
    "normal_disable_rate": normal_disable_rate,
    "selection_accuracy": selection_accuracy,
    "greedy_total_reward": total_reward,
}])
metrics_df.to_csv(metrics_path, index=False)

print("trace saved:", trace_path)
print("metrics saved:", metrics_path)
print(metrics_df)
print(trace_df[["rule_idx", "action", "target_label", "is_correct"]])

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
win_id = 1435

metrics_path = ROOT / "outputs/logs" / f"window_{win_id}_metrics.csv"
summary_path = ROOT / "outputs/logs" / "mixed_windows_summary_with_ratio_v1.csv"

df_metrics = pd.read_csv(metrics_path)

df_metrics["attack_ratio"] = 1690 / 2000  # window 1435 当前窗口标签统计得到
cols = [
    "window_id", "attack_ratio",
    "num_rules", "num_attack_rules", "num_normal_rules",
    "attack_keep_rate", "normal_disable_rate",
    "selection_accuracy", "greedy_total_reward"
]
new_row = df_metrics[cols].copy()

if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    df_summary = df_summary[df_summary["window_id"] != win_id].copy()
    df_summary = pd.concat([df_summary, new_row], axis=0, ignore_index=True)
else:
    df_summary = new_row.copy()

df_summary = df_summary.sort_values("window_id").reset_index(drop=True)
df_summary.to_csv(summary_path, index=False)

print("saved:", summary_path)
print(df_summary)
print("\nmean selection_accuracy:", df_summary["selection_accuracy"].mean())
print("mean attack_keep_rate:", df_summary["attack_keep_rate"].mean())
print("mean normal_disable_rate:", df_summary["normal_disable_rate"].mean())

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
summary_path = ROOT / "outputs/logs" / "mixed_windows_summary_with_ratio_v1.csv"
save_path = ROOT / "outputs/figures" / "attack_ratio_vs_selection_accuracy_v2.png"

df = pd.read_csv(summary_path).copy()
df = df.sort_values("attack_ratio").reset_index(drop=True)

plt.figure(figsize=(6, 4))
plt.scatter(df["attack_ratio"], df["selection_accuracy"], s=60)

for _, row in df.iterrows():
    plt.text(
        row["attack_ratio"] + 0.005,
        row["selection_accuracy"] + 0.002,
        str(int(row["window_id"])),
        fontsize=9
    )

plt.xlabel("attack_ratio")
plt.ylabel("selection_accuracy")
plt.title("Attack Ratio vs Selection Accuracy")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", save_path)
print(df[["window_id", "attack_ratio", "selection_accuracy", "normal_disable_rate"]])